In [ ]:
import multiprocessing as mp
import time
import os
import sys

def f(n):
    time.sleep(2)
    print(f"{os.getpid()} ha ricevuto il valore numero {n}")

def main():
    # In notebook su macOS, fork evita molti problemi legati a spawn.
    ctx = mp.get_context("fork") if sys.platform == "darwin" else mp.get_context()
    procs = []
    for i in range(10):
        p = ctx.Process(target=f, args=(i,))
        p.start()
        procs.append(p)

    for p in procs:
        p.join()

if __name__ == '__main__':
    main()

9334 ha ricevuto il valore numero 0
9335 ha ricevuto il valore numero 1
9336 ha ricevuto il valore numero 2
9337 ha ricevuto il valore numero 3
9338 ha ricevuto il valore numero 4
9339 ha ricevuto il valore numero 5
9340 ha ricevuto il valore numero 6
9341 ha ricevuto il valore numero 7
9342 ha ricevuto il valore numero 8
9343 ha ricevuto il valore numero 9


In [7]:
import multiprocessing as mp
import time
import os
import sys

def f(q, n):
    time.sleep(2)
    print(f"{os.getpid()} mette in coda il numero {n}")
    q.put(n)

def main():
    ctx = mp.get_context("fork") if sys.platform == "darwin" else mp.get_context()
    procs = []
    q = ctx.Queue()

    for i in range(10):
        p = ctx.Process(target=f, args=(q, i))
        p.start()
        procs.append(p)

    for _ in range(10):
        n = q.get()
        print(f"ricevuto {n}")

    for p in procs:
        p.join()

if __name__ == '__main__':
    main()

9495 mette in coda il numero 0
9496 mette in coda il numero 1
9497 mette in coda il numero 2
9498 mette in coda il numero 3
9499 mette in coda il numero 4
9500 mette in coda il numero 5
9501 mette in coda il numero 6
9502 mette in coda il numero 7
9503 mette in coda il numero 8
9504 mette in coda il numero 9
ricevuto 0
ricevuto 1
ricevuto 2
ricevuto 3
ricevuto 4
ricevuto 5
ricevuto 6
ricevuto 7
ricevuto 8
ricevuto 9


multiprocessing python:
pool: processi delegare compiti


In [ ]:
import os
def(a,b,e):
    for i in range(b,e):
        a[i] = a[i] ** 2
        print(f"{os.getpid()} ha completato il lavoro")
    
        
        
def main():
    print(f"Processo principale {os.getpid()}")
    a = mp.Array('i', [i for i in range(100)])
    procs = []
    for i in range(10):
        p = mp.Process(target=f, args=(a, i*10, (i+1)*10))
        procs.append(p)
        p.start()
    for p in procs:
        p.join()
        lst = [a[i] for i in range(100)]
        print(lst)

In [10]:
import multiprocessing as mp
import time
import os

def f(n):
    time.sleep(2)
    print(f"{os.getpid()} ha ricevuto il valore numero {n}")

def main():
    p = mp.Pool(processes=4)
    p.map(f, range(10))

if __name__ == '__main__':
    main()

Process SpawnPoolWorker-43:
Process SpawnPoolWorker-42:
Process SpawnPoolWorker-44:
Process SpawnPoolWorker-41:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.11/3.11.15/Frameworks/Python.framework/Versions/3.11/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/homebrew/Cellar/python@3.11/3.11.15/Frameworks/Python.framework/Versions/3.11/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/homebrew/Cellar/python@3.11/3.11.15/Frameworks/Python.framework/Versions/3.11/lib/python3.11/multiprocessing/pool.py", line 114, in worker
    task = get()
           ^^^^^
  File "/opt/homebrew/Cellar/python@3.11/3.11.15/Frameworks/Python.framework/Versions/3.11/lib/python3.11/multiprocessing/queues.py", line 367, in get
    return _ForkingPickler.loads(res)
 

KeyboardInterrupt: 

just in time compilation: Numba
- si spende tempo a compilare, se eseguo + vole è piu veloce, uso lib numba usa LLVM
- uso decoratore posso compilare in codice nativo: @jit



In [9]:
from numba import jit
import timeit
import random

@jit
def f(n):
    res = 0
    for i in range(n):
        res += i**2 + 2*i + 1
    return res

def g(n):
    res = 0
    for i in range(n):
        res += i**2 + 2*i + 1
    return res
@jit
def f_slow(lst):
    res = ""
    for x in lst:
        res += str(x)
    return len(res)

def g_slow(lst):
    res = ""
    for x in lst:
        res += str(x)
    return len(res)

print(f"Eseguzione di g: {timeit.timeit('lambda: g(100)')} secondi")
print(f"Eseguzione di f: {timeit.timeit('lambda: f(100)')} secondi")
print(f"Esecuzione di g_slow: {timeit.timeit('lambda: g_slow(list(range(100000000)))')} secondi")
print(f"Esecuzione di f_slow: {timeit.timeit('lambda: f_slow(list(range(100000000)))')} secondi")

Eseguzione di g: 0.05601999999998952 secondi
Eseguzione di f: 0.042182167000646587 secondi
Esecuzione di g_slow: 0.04172362500048621 secondi
Esecuzione di f_slow: 0.0411862500004645 secondi


In [17]:
from numba import vectorize, float64, jit
import timeit
import numpy as np

def standard_fma(va, vb, vc):
    out = np.empty_like(va)
    for i in range(va.shape[0]):
        out[i] = va[i] * vb[i] + vc[i]
    return out

@jit
def jit_fma(va, vb, vc):
    out = np.empty_like(va)
    for i in range(va.shape[0]):
        out[i] = va[i] * vb[i] + vc[i]
    return out

@vectorize([float64(float64, float64, float64)])
def vectorized_fma(a, b, c):
    return a * b + c

n = 1_000_000
rng = np.random.default_rng()
a = rng.random(n)
b = rng.random(n)
c = rng.random(n)

print(f"Standard FMA: {timeit.timeit(lambda: standard_fma(a, b, c), number=1)}")
print(f"JIT FMA: {timeit.timeit(lambda: jit_fma(a, b, c), number=1)}")
print(f"Vectorized FMA: {timeit.timeit(lambda: vectorized_fma(a, b, c), number=1)}")


Standard FMA: 0.21136495799964905
JIT FMA: 0.04759799999919778
Vectorized FMA: 0.0004820000003746827


@guvectorize
vectorize funziona bene se abiamo stesse dim di input tutti gli array
alcuni argomenti rimanare scalari output diventa differente
+ generico:  guvectorize

paralelizzare cii for: @jit(parallel=True)
    usa prange, numba fornsice prange

In [ ]:
import numpy as np

def matmul(M1,M2):
    Mout = np.zeros_like(M1)
    n = M1.shape[0]
    for i in range(n):
        for j in range(n):
            for k in range(n):
                Mout[i,j] += M1[i,k] * M2[k,j]
    return Mout

@jit
def matmul(M1,M2):
    Mout = np.zeros_like(M1)
    n = M1.shape[0]
    for i in range(n):
        for j in range(n):
            for k in range(n):
                Mout[i,j] += M1[i,k] * M2[k,j]
    return Mout




n = 250
rng = np.random.default_rng()
M1 = rng.random((n,n))
M2 = rng.random((n,n))

print(f"Matmul: {timeit.timeit(lambda: matmul(M1, M2), number=1)} secondi")
print(f"JIT Matmul: {timeit.timeit(lambda: matmul(M1, M2), number=1)} secondi")

Matmul: 0.08524779099934676 secondi
JIT Matmul: 0.013383499999690684 secondi
